# ODI to Databricks Migration

## Source File: TRG_EMP_INSERT.txt
## Conversion Timestamp: 2024-07-30T12:00:00Z

### Description: This notebook converts an ODI scenario to insert data into the TRG_EMP table from EMPLOYEES table. (SCEN_TASK_NO in {10}, SCEN_TASK_NO in {20})

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "")
dbutils.widgets.text("DATASOURCE_NUM_ID", "")
dbutils.widgets.text("ETL_PROC_WID", "")
dbutils.widgets.text("ODI_SESS_NO", "")
dbutils.widgets.text("GLOBAL_V_ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00")
dbutils.widgets.text("GLOBAL_V_ETL_CURRENT_EXTRACT_TIME", "9999-12-31 23:59:59")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS SELECT '${ETL_JOB_TYPE}' AS etl_job_type;
CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS SELECT ${DATASOURCE_NUM_ID} AS datasource_num_id;
CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS SELECT ${ETL_PROC_WID} AS etl_proc_wid;
CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS SELECT '${ODI_SESS_NO}' AS odi_sess_no;
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS SELECT to_timestamp('${GLOBAL_V_ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time;
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS SELECT to_timestamp('${GLOBAL_V_ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("""
  SELECT
    (SELECT etl_job_type FROM v_etl_job_type) AS ETL_JOB_TYPE,
    (SELECT datasource_num_id FROM v_datasource_num_id) AS DATASOURCE_NUM_ID,
    (SELECT etl_proc_wid FROM v_etl_proc_wid) AS ETL_PROC_WID,
    (SELECT odi_sess_no FROM v_odi_sess_no) AS ODI_SESS_NO,
    (SELECT etl_last_extract_time FROM v_etl_last_extract_time) AS ETL_LAST_EXTRACT_TIME,
    (SELECT etl_current_extract_time FROM v_etl_current_extract_time) AS ETL_CURRENT_EXTRACT_TIME
"""))

## Load Target Table `trg_emp`

In [ ]:
%sql
-- SCEN_TASK_NO in {30}
INSERT INTO workspace.hr.trg_emp
  (
    employee_id ,
    first_name ,
    last_name ,
    email ,
    phone_number ,
    hire_date ,
    job_id ,
    salary ,
    commission_pct ,
    manager_id ,
    department_id
  )
SELECT
  employees.employee_id ,
  employees.first_name ,
  employees.last_name ,
  employees.email ,
  employees.phone_number ,
  employees.hire_date ,
  employees.job_id ,
  employees.salary ,
  employees.commission_pct ,
  employees.manager_id ,
  employees.department_id
FROM
  workspace.hr.employees AS employees;


## Validation

In [ ]:
%sql
SELECT COUNT(*) FROM workspace.hr.trg_emp;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names**: All Oracle schema and table names have been converted to lowercase and prefixed with `workspace.` (e.g., `HR.TRG_EMP` -> `workspace.hr.trg_emp`).
2.  **Oracle Hints**: The Oracle hint `/*+ APPEND PARALLEL */` has been removed as it is not applicable in Databricks Spark SQL.
3.  **Widgets**: Standard ETL parameter widgets (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`, `GLOBAL_V_ETL_LAST_EXTRACT_TIME`, `GLOBAL_V_ETL_CURRENT_EXTRACT_TIME`) and their corresponding temporary views have been added for consistency with the migration framework, even though they are not directly referenced in the provided SQL `INSERT` statement.
4.  **SCEN_TASK_NO**: The `SCEN_TASK_NO` markers {10}, {20}, and {30} are noted as comments within the notebook to maintain traceability of the original ODI flow.